1. 
사용자와 컴퓨터가 가위바위보A 진행

2. 
A에서 한쪽이 승리할 때까지 A를 반복 진행

3. 
사용자가 A에서 이긴 경우: X = 가위바위보 B를 진행 (사용자가 공격권자)
user1. X에서 이긴 경우 X를 다시 진행
user2. X에서 진 경우 Y 진행
user3. X에서 비긴 경우 > 사용자 승, 컴퓨터 패

4. 
컴퓨터가 A에서 이긴 경우: Y = 가위바위보 C를 진행 (컴퓨터가 공격권자)
computer1. Y에서 이긴 경우 -> Y를 다시 진행
computer2. Y에서 진 경우 X 진행
computer3. Y에서 비긴 경우 > 컴퓨터 승, 사용자 패

In [ ]:
import random
from enum import Enum

class Hand(Enum):
    """가위바위보에서 낼 수 있는 손모양"""
    ROCK = "바위"
    SCISSORS = "가위"
    PAPER = "보"

    def beats(self, other: "Hand") -> bool:
        """self가 other를 이기면 True를 반환한다."""
        winning_pairs = {
            (Hand.ROCK, Hand.SCISSORS),
            (Hand.SCISSORS, Hand.PAPER),
            (Hand.PAPER, Hand.ROCK),
        }
        return (self, other) in winning_pairs

    def to_mukjjippa(self) -> str:
        """묵찌빠 용어로 바꿔준다. (바위->묵, 가위->찌, 보->빠)"""
        mapping = {Hand.ROCK: "묵", Hand.SCISSORS: "찌", Hand.PAPER: "빠"}
        return mapping[self]

class Player:
    """플레이어의 공통 동작을 정의하는 기본 클래스."""

    def __init__(self, name: str):
        self.name = name

    def choose_hand(self, prompt: str = "") -> Hand:
        raise NotImplementedError("자식 클래스에서 구현해야 합니다.")

class HumanPlayer(Player):
    """키보드 입력을 받아 손을 내는 사람 플레이어."""

    def choose_hand(self, prompt: str = "") -> Hand:
        options = {"1": Hand.ROCK, "2": Hand.SCISSORS, "3": Hand.PAPER}
        while True:
            choice = input(f"{prompt} (1: 바위, 2: 가위, 3: 보) >> ").strip()
            if choice in options:
                return options[choice]
            print("잘못된 입력입니다. 1, 2, 3 중에서 골라주세요.")

class ComputerPlayer(Player):
    """무작위로 손을 내는 컴퓨터 플레이어."""

    def choose_hand(self, prompt: str = "") -> Hand:
        return random.choice(list(Hand))

class MukjjippaGame:
    """묵찌빠 게임 전체 진행을 담당하는 클래스."""

    def __init__(self, player1: Player, player2: Player):
        self.player1 = player1
        self.player2 = player2

    def play_rock_paper_scissors(self):
        """선공을 정하기 위한 가위바위보를 진행하고 (공격자, 수비자)를 반환한다."""
        round_num = 1
        while True:
            print(f"\n[가위바위보 {round_num}회차] 선공을 정합니다.")
            hand1 = self.player1.choose_hand(f"{self.player1.name}, 손을 내세요")
            hand2 = self.player2.choose_hand(f"{self.player2.name}, 손을 내세요")
            print(f"{self.player1.name}: {hand1.value}  /  {self.player2.name}: {hand2.value}")
 
            if hand1 == hand2:
                print("비겼습니다! 다시 냅니다.")
                round_num += 1
                continue
 
            if hand1.beats(hand2):
                print(f"{self.player1.name} 승리! 선공(공격)을 잡았습니다.")
                return self.player1, self.player2
            else:
                print(f"{self.player2.name} 승리! 선공(공격)을 잡았습니다.")
                return self.player2, self.player1
 
    def play_mukjjippa_round(self, attacker: Player, defender: Player):
        """묵찌빠 한 판을 진행한다. 승부가 났으면 None, 아니면 다음 공격자를 반환한다."""
        attacker_hand = attacker.choose_hand(f"[공격] {attacker.name}, 손을 내세요")
        defender_hand = defender.choose_hand(f"[수비] {defender.name}, 손을 내세요")
 
        print(
            f"{attacker.name}(공격): {attacker_hand.to_mukjjippa()}  /  "
            f"{defender.name}(수비): {defender_hand.to_mukjjippa()}"
        )
 
        if attacker_hand == defender_hand:
            print(f"\n같은 손이 나왔습니다! {attacker.name}의 승리로 게임이 끝났습니다.")
            return None
 
        if attacker_hand.beats(defender_hand):
            print(f"{attacker.name}가 이겨서 계속 공격합니다.")
            return attacker
        else:
            print(f"{defender.name}가 이겨서 공격권을 가져옵니다.")
            return defender
 
    def play(self):
        print("=" * 40)
        print("        묵 찌 빠  게 임  시 작")
        print("=" * 40)
 
        attacker, defender = self.play_rock_paper_scissors()
        print(f"\n이제부터 {attacker.name}가 공격, {defender.name}가 수비입니다.")
        print("공격자와 같은 손을 내면 게임이 끝나요. 다르면 이긴 쪽이 공격권을 가져갑니다.\n")
 
        while True:
            next_attacker = self.play_mukjjippa_round(attacker, defender)
            if next_attacker is None:
                break
            if next_attacker is defender:
                attacker, defender = defender, attacker
            # next_attacker가 attacker와 같으면 그대로 유지
 
        print(f"\n최종 승자는 {attacker.name} 입니다.")
 
 
def main():
    print("묵찌빠 게임에 오신 것을 환영합니다!")
    name = input("당신의 이름을 입력하세요: ").strip() or "플레이어"
 
    human = HumanPlayer(name)
    computer = ComputerPlayer("컴퓨터")
 
    game = MukjjippaGame(human, computer)
    game.play()
 
 
if __name__ == "__main__":
    main()